# Session 2 · Part 4 — Run the pretrained inference handoff

**Goal:** use a verified DGAT artifact after learning how it was built and trained. Conference laptops
load committed official predictions for speed and reproducibility; the organizer command below can
regenerate them from the RNA graph and saved RNA-encoder/protein-decoder weights.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Inspect prediction provenance before trusting values


In [ ]:
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table, write_prediction_artifact

source_path = paths.raw_data / "dgat_predictions.csv"
if not source_path.is_file():
    raise FileNotFoundError(f"Missing verified prediction table: {source_path}")
predicted_proteins = load_prediction_table(str(source_path))
metadata = load_prediction_metadata(source_path)
if metadata is None:
    raise FileNotFoundError(f"Missing provenance sidecar for {source_path}")
display(metadata, predicted_proteins.head())


## 2. The inference operation represented by this artifact

```python
z_rna = encoder_rna(x_rna, rna_edge_index)
predicted_protein = decoder_protein(z_rna)
```

To regenerate with official assets, run `scripts/run_official_dgat_prediction.py` in the separate official
environment. That script discovers the matching common-feature lists and checkpoint layout, then calls
upstream `Model.Train_and_Predict.protein_predict`.


In [ ]:
output_path = paths.processed_data / "predicted_proteins.csv"
metadata_path = write_prediction_artifact(
    predicted_proteins, output_path,
    method=str(metadata["method"]), source=str(metadata["source"]),
    evaluation_note=str(metadata["evaluation_note"]),
)
manifest = write_checkpoint(
    "2.4", [output_path, metadata_path],
    summary={"spots": len(predicted_proteins), "proteins": predicted_proteins.shape[1]}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Do not evaluate a model on protein values that were used to train or select it. Read the sidecar's
evaluation note and confirm that spot IDs and protein names match the target dataset.
